In [1]:
import pandas as pd


In [2]:
columns = ['PublicID', 'rest_dur_avg_all_Mod', 'rest_sleeptime_avg_all_Mod', 'sleep_dur_avg_all_Mod', 'sleep_sleeptime_avg_all_Mod', 'sleep_Frag_avg_all_Mod', 'sleep_WASO_avg_all_Mod', 'sleep_SE_avg_all_Mod', 'rest_sleeptime_avg_wkday_Mod']  

df_in = pd.read_csv('SLEEP_ACTIGRAPHY_MODIFIED.CSV', usecols=columns)


In [3]:
df_in

,PublicID,rest_dur_avg_all_Mod,rest_sleeptime_avg_all_Mod,rest_sleeptime_avg_wkday_Mod,sleep_dur_avg_all_Mod,sleep_sleeptime_avg_all_Mod,sleep_Frag_avg_all_Mod,sleep_WASO_avg_all_Mod,sleep_SE_avg_all_Mod
0,00022M,478.916656,413.416656,399.899994,467.916656,409.666656,26.400000,58.250000,85.633331
1,00037W,577.857117,512.785706,489.299988,572.214294,510.928558,16.444286,61.285713,88.507149
2,00051F,615.900024,519.500000,537.833313,607.000000,518.500000,29.483999,88.500000,84.447998
3,00053B,460.714294,399.714294,420.000000,453.142853,396.785706,24.460001,56.357143,83.424278
4,00062A,657.857117,553.785706,578.700012,631.642883,545.142883,36.068573,86.500000,83.761429
...,...,...,...,...,...,...,...,...,...
653,17298W,484.333344,423.583344,421.000000,476.916656,422.000000,18.618334,54.916668,87.174995
654,17317V,536.642883,455.214294,434.100006,523.785706,451.714294,25.052856,72.071426,84.125717
655,17343U,535.071411,465.928558,434.000000,525.857117,465.500000,20.960001,60.357143,86.921432
656,17351V,655.000000,574.333313,536.125000,645.833313,569.583313,24.150002,76.250000,86.904999


In [4]:
df_out = pd.read_csv('Outcome.csv',usecols=['V4AH01','PublicID'])
df = pd.merge(df_in, df_out, on='PublicID', how='inner')


In [5]:
df = df.dropna()

In [6]:
df.to_csv('sleep_df.csv', index=False)


In [20]:
X = df.drop(['PublicID','V4AH01'],axis=1)
y = df['V4AH01']


In [26]:
y

2      1.0
3      1.0
5      1.0
6      3.0
7      1.0
      ... 
590    1.0
591    1.0
593    3.0
594    1.0
596    3.0
Name: V4AH01, Length: 419, dtype: float64

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X = pd.DataFrame(X_scaled, columns=X.columns)



In [22]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE


X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

model = LogisticRegression(solver='liblinear')

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Model Coefficients:")
for feature, coef in zip(X.columns, model.coef_[0]):
    print(f"{feature}: {coef}")


Model Coefficients:
rest_dur_avg_all_Mod: -0.49626599240263986
rest_sleeptime_avg_all_Mod: -0.024813185533966768
rest_sleeptime_avg_wkday_Mod: -0.385255556158815
sleep_dur_avg_all_Mod: 0.4628878318434267
sleep_sleeptime_avg_all_Mod: 0.6126450598267785
sleep_Frag_avg_all_Mod: -0.1688513090566332
sleep_WASO_avg_all_Mod: -0.19469700192252629
sleep_SE_avg_all_Mod: 0.18044041559056637


In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Compute evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred,average='weighted')
recall = recall_score(y_test, y_pred,average='weighted')
f1 = f1_score(y_test, y_pred,average='weighted')
# roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

# Print the evaluation metrics
print("Evaluation Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
# print(f"ROC AUC: {roc_auc:.4f}")

Evaluation Metrics:
Accuracy: 0.4762
Precision: 0.5308
Recall: 0.4762
F1-score: 0.5003


In [23]:
# from sklearn.ensemble import RandomForestClassifier

# rf_model = RandomForestClassifier(random_state=42)

# rf_model.fit(X_train, y_train)

# y_pred = rf_model.predict(X_test)

# accuracy = accuracy_score(y_test, y_pred)
# precision = precision_score(y_test, y_pred, average='weighted')
# recall = recall_score(y_test, y_pred, average='weighted')
# f1 = f1_score(y_test, y_pred, average='weighted')
# # roc_auc = roc_auc_score(y_test, y_pred)

# print("Evaluation Metrics:")
# print(f"Accuracy: {accuracy:.4f}")
# print(f"Precision: {precision:.4f}")
# print(f"Recall: {recall:.4f}")
# print(f"F1-score: {f1:.4f}")
# # print(f"ROC AUC: {roc_auc:.4f}")



Evaluation Metrics:
Accuracy: 0.4762
Precision: 0.5075
Recall: 0.4762
F1-score: 0.4893
